# Optuna Study with PatchCore and PDT Dataset to Collect Logs 

In [1]:
import torch

GPU_ID = 1

if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    torch.cuda.set_per_process_memory_fraction(0.7, device=GPU_ID)
    DEVICE = torch.device(f"cuda:{GPU_ID}")
else:
    DEVICE = torch.device("cpu")

print("device_count:", torch.cuda.device_count() if torch.cuda.is_available() else 0)
print("current_device:", torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
print("using:", DEVICE)

device_count: 2
current_device: 1
using: cuda:1


In [2]:
# =========================
# 0) Setup: imports & config
# =========================
from __future__ import annotations

from pathlib import Path
import random
import json
import time
import zlib

import numpy as np
import pandas as pd
import optuna
import torch
import gc

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image, ImageFilter

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from sklearn.neighbors import NearestNeighbors

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("SEED:", SEED)

SEED: 42


In [3]:
# =========================
# 1) Paths: locate ROOT and dataset folders
# =========================

ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "fast").is_dir() and (p / "shared").is_dir()
)

DATASET_DIR = Path("/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection")

assert DATASET_DIR.is_dir(), f"Missing dataset dir: {DATASET_DIR}"

print("ROOT:", ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("Classes found:", sorted([p.name for p in DATASET_DIR.iterdir() if p.is_dir()]))

ROOT: /home/jovyan
DATASET_DIR: /home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection
Classes found: ['bottle', 'cable', 'capsule', 'carpet', 'coins', 'grid', 'hazelnut', 'leather', 'logs', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [4]:
SELECTED_CLASSES = ["bottle", "cable", "capsule", "hazelnut"]

print("SELECTED_CLASSES:", SELECTED_CLASSES)

SELECTED_CLASSES: ['bottle', 'cable', 'capsule', 'hazelnut']


In [5]:
def list_imgs(folder: Path) -> list[Path]:
    exts = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.PNG", "*.JPG", "*.JPEG", "*.BMP"]
    paths = []
    for ext in exts:
        paths.extend(folder.rglob(ext))
    return sorted(set(paths))

In [6]:
def build_mvtec_class_frames(class_name: str, dataset_dir: Path, seed: int = 42):
    class_dir = dataset_dir / class_name

    train_good_dir = class_dir / "train" / "good"
    test_dir = class_dir / "test"

    assert train_good_dir.is_dir(), f"Missing train/good for {class_name}: {train_good_dir}"
    assert test_dir.is_dir(), f"Missing test dir for {class_name}: {test_dir}"

    train_good_paths = list_imgs(train_good_dir)
    assert len(train_good_paths) > 0, f"No train/good images found for {class_name}"

    df_train_good = pd.DataFrame({
        "path": [str(p) for p in train_good_paths],
        "label": 0,
        "mvtec_class": class_name,
        "split": "train_good",
        "defect_type": "good",
    })

    test_rows = []
    defect_dirs = sorted([p for p in test_dir.iterdir() if p.is_dir()])

    for defect_dir in defect_dirs:
        defect_type = defect_dir.name
        label = 0 if defect_type == "good" else 1
        paths = list_imgs(defect_dir)

        for p in paths:
            test_rows.append({
                "path": str(p),
                "label": label,
                "mvtec_class": class_name,
                "split": "test",
                "defect_type": defect_type,
            })

    df_test = pd.DataFrame(test_rows)
    assert len(df_test) > 0, f"No test images found for {class_name}"
    assert df_test["label"].nunique() == 2, f"Need both good and anomalous test samples for {class_name}"

    df_val, df_final = train_test_split(
        df_test,
        test_size=0.5,
        random_state=seed,
        stratify=df_test["label"],
    )

    return (
        df_train_good.reset_index(drop=True),
        df_val.reset_index(drop=True),
        df_final.reset_index(drop=True),
        df_test.reset_index(drop=True),
    )

In [7]:
CLASS_DATA = {}

for cls in SELECTED_CLASSES:
    df_train_good, df_val, df_final, df_test = build_mvtec_class_frames(
        class_name=cls,
        dataset_dir=DATASET_DIR,
        seed=SEED,
    )

    CLASS_DATA[cls] = {
        "df_train_good": df_train_good,
        "df_val": df_val,
        "df_final": df_final,
        "df_test": df_test,
    }

    print(f"\nClass: {cls}")
    print(" train_good:", len(df_train_good))
    print(" val labels:", df_val["label"].value_counts().to_dict())
    print(" final labels:", df_final["label"].value_counts().to_dict())


Class: bottle
 train_good: 209
 val labels: {1: 31, 0: 10}
 final labels: {1: 32, 0: 10}

Class: cable
 train_good: 224
 val labels: {1: 46, 0: 29}
 final labels: {1: 46, 0: 29}

Class: capsule
 train_good: 219
 val labels: {1: 55, 0: 11}
 final labels: {1: 54, 0: 12}

Class: hazelnut
 train_good: 391
 val labels: {1: 35, 0: 20}
 final labels: {1: 35, 0: 20}


### Data loading

In [8]:
SPLIT_OUT = ROOT / "fast" / "x_peer_pdt_hyperoptxai" / "hpo_mvtec_4class" / "mvtec_splits"
SPLIT_OUT.mkdir(parents=True, exist_ok=True)

split_meta = {
    "seed": SEED,
    "dataset_dir": str(DATASET_DIR),
    "selected_classes": SELECTED_CLASSES,
    "classes": {},
}

for cls in SELECTED_CLASSES:
    cls_out = SPLIT_OUT / cls
    cls_out.mkdir(parents=True, exist_ok=True)

    CLASS_DATA[cls]["df_train_good"].to_csv(cls_out / "df_train_good.csv", index=False)
    CLASS_DATA[cls]["df_val"].to_csv(cls_out / "df_val.csv", index=False)
    CLASS_DATA[cls]["df_final"].to_csv(cls_out / "df_final.csv", index=False)

    split_meta["classes"][cls] = {
        "train_good": int(len(CLASS_DATA[cls]["df_train_good"])),
        "val": int(len(CLASS_DATA[cls]["df_val"])),
        "final": int(len(CLASS_DATA[cls]["df_final"])),
        "val_labels": CLASS_DATA[cls]["df_val"]["label"].value_counts().to_dict(),
        "final_labels": CLASS_DATA[cls]["df_final"]["label"].value_counts().to_dict(),
    }

with open(SPLIT_OUT / "split_meta.json", "w", encoding="utf-8") as f:
    json.dump(split_meta, f, indent=2)

print("Exported splits to:", SPLIT_OUT)

Exported splits to: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits


### PatchCore core: feature extractor + embedding

In [9]:
SOFT_TRAIN_FRACTION_RANGE = (0.2, 1.0)
SOFT_CORRUPTION_LEVELS = ["none", "mild", "strong"]
SOFT_REVIEW_BUDGETS = [i / 1000 for i in range(5, 501, 5)]

print("SOFT_TRAIN_FRACTION_RANGE:", SOFT_TRAIN_FRACTION_RANGE)
print("SOFT_CORRUPTION_LEVELS:", SOFT_CORRUPTION_LEVELS)
print("SOFT_REVIEW_BUDGETS:", SOFT_REVIEW_BUDGETS[:5], "...", SOFT_REVIEW_BUDGETS[-5:])

SOFT_TRAIN_FRACTION_RANGE: (0.2, 1.0)
SOFT_CORRUPTION_LEVELS: ['none', 'mild', 'strong']
SOFT_REVIEW_BUDGETS: [0.005, 0.01, 0.015, 0.02, 0.025] ... [0.48, 0.485, 0.49, 0.495, 0.5]


In [10]:
def _stable_int_seed(text: str, seed: int) -> int:
    return (zlib.crc32(text.encode("utf-8")) + seed) & 0xFFFFFFFF


def corrupt_pil(img: Image.Image, level: str, seed_int: int) -> Image.Image:
    if level == "none":
        return img

    rng = np.random.default_rng(seed_int)
    out = img

    if level in {"mild", "strong"}:
        radius = 0.6 if level == "mild" else 1.2
        out = out.filter(ImageFilter.GaussianBlur(radius=radius))

    if level == "strong":
        arr = np.asarray(out).astype(np.float32)
        noise_std = 8.0
        noise = rng.normal(0.0, noise_std, size=arr.shape).astype(np.float32)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)

    return out


def q01(x: float, lo: float, hi: float) -> float:
    x = max(lo, min(hi, float(x)))
    return round(x * 100.0) / 100.0

In [11]:
class ImagePathDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        preproc: transforms.Compose,
        corruption_level: str = "none",
        apply_corruption: bool = False,
    ):
        self.paths = df["path"].tolist()
        self.labels = df["label"].to_numpy().astype(int)
        self.preproc = preproc
        self.corruption_level = corruption_level
        self.apply_corruption = apply_corruption

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")

        if self.apply_corruption and self.corruption_level != "none":
            s = _stable_int_seed(path, SEED)
            img = corrupt_pil(img, self.corruption_level, s)

        x = self.preproc(img)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [12]:
def make_preproc(image_size: tuple[int, int], center_crop_size: tuple[int, int] | None):
    t = [transforms.Resize(image_size)]
    if center_crop_size is not None:
        t.append(transforms.CenterCrop(center_crop_size))
    t += [
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
    return transforms.Compose(t)


def make_loaders(
    df_train_good_sub: pd.DataFrame,
    df_val: pd.DataFrame,
    df_final: pd.DataFrame,
    image_size: tuple[int, int],
    center_crop_size: tuple[int, int] | None,
    batch_size: int,
    corruption_level: str,
):
    preproc = make_preproc(image_size=image_size, center_crop_size=center_crop_size)

    train_ds = ImagePathDataset(
        df_train_good_sub,
        preproc,
        corruption_level=corruption_level,
        apply_corruption=True,
    )
    val_ds = ImagePathDataset(
        df_val,
        preproc,
        corruption_level="none",
        apply_corruption=False,
    )
    final_ds = ImagePathDataset(
        df_final,
        preproc,
        corruption_level="none",
        apply_corruption=False,
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    final_loader = DataLoader(final_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, final_loader

In [13]:
def build_backbone(backbone: str, pre_trained: bool) -> torch.nn.Module:
    if backbone == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pre_trained else None)
    elif backbone == "wide_resnet50_2":
        m = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT if pre_trained else None)
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")

    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    return m

In [14]:
class FeatureExtractor(torch.nn.Module):
    def __init__(self, backbone: str, layers: tuple[str, ...], pre_trained: bool):
        super().__init__()
        self.backbone = build_backbone(backbone, pre_trained)
        self.layers = layers
        self._features: dict[str, torch.Tensor] = {}

        named = dict(self.backbone.named_modules())
        for name in layers:
            if name not in named:
                raise ValueError(f"Layer '{name}' not found in backbone '{backbone}'")
            named[name].register_forward_hook(self._hook(name))

    def _hook(self, name: str):
        def fn(module, inp, out):
            self._features[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        _ = self.backbone(x)
        return [self._features[name] for name in self.layers]

In [15]:
def to_patch_embeddings(
    feat_maps: list[torch.Tensor],
    max_patches_per_image: int | None = None,
    seed: int = 0,
) -> torch.Tensor:
    H = max(f.shape[-2] for f in feat_maps)
    W = max(f.shape[-1] for f in feat_maps)

    resized = []
    for f in feat_maps:
        if (f.shape[-2], f.shape[-1]) != (H, W):
            f = torch.nn.functional.interpolate(f, size=(H, W), mode="bilinear", align_corners=False)
        resized.append(f)

    f_cat = torch.cat(resized, dim=1)   # [B, D, H, W]
    B, D, _, _ = f_cat.shape

    f_flat = f_cat.flatten(2).transpose(1, 2).contiguous()   # [B, P, D]
    P = f_flat.shape[1]

    if max_patches_per_image is not None and max_patches_per_image < P:
        g = torch.Generator(device=f_flat.device)
        g.manual_seed(seed)
        idx = torch.randperm(P, generator=g, device=f_flat.device)[:max_patches_per_image]
        f_flat = f_flat[:, idx, :]

    return f_flat.reshape(-1, D)

In [16]:
def coreset_sample(embeddings: np.ndarray, ratio: float, seed: int) -> np.ndarray:
    n = embeddings.shape[0]
    m = max(1, int(n * ratio))
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=m, replace=False)
    return embeddings[idx]

In [17]:
@torch.no_grad()
def fit_memory_bank(
    extractor: FeatureExtractor,
    train_loader: DataLoader,
    coreset_sampling_ratio: float,
    max_patches_per_image: int,
) -> np.ndarray:
    extractor = extractor.to(DEVICE).eval()

    kept = []
    for x, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = to_patch_embeddings(
            feats,
            max_patches_per_image=max_patches_per_image,
            seed=SEED,
        )
        kept.append(patches.detach().cpu().numpy())

        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    emb = np.vstack(kept)
    return coreset_sample(emb, ratio=coreset_sampling_ratio, seed=SEED)

In [18]:
@torch.no_grad()
def score_images(
    extractor: FeatureExtractor,
    loader: DataLoader,
    memory_bank: np.ndarray,
    num_neighbors: int,
    reduction: str,
    max_patches_per_image: int,
) -> tuple[np.ndarray, np.ndarray]:
    extractor = extractor.to(DEVICE).eval()

    nn = NearestNeighbors(n_neighbors=num_neighbors, algorithm="auto")
    nn.fit(memory_bank)

    y_true_list, y_score_list = [], []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = to_patch_embeddings(
            feats,
            max_patches_per_image=max_patches_per_image,
            seed=SEED,
        ).detach().cpu().numpy()

        dists, _ = nn.kneighbors(patches, return_distance=True)
        patch_scores = dists.mean(axis=1)

        B = y.shape[0]
        P = patch_scores.shape[0] // B
        patch_scores = patch_scores.reshape(B, P)

        img_scores = patch_scores.max(axis=1) if reduction == "max" else patch_scores.mean(axis=1)

        y_true_list.append(y.numpy())
        y_score_list.append(img_scores)

        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(y_true_list), np.concatenate(y_score_list)

In [19]:
def threshold_from_review_budget(y_score: np.ndarray, budget: float) -> float:
    assert 0.0 < budget < 1.0
    n = y_score.shape[0]
    k = max(1, int(np.ceil(budget * n)))
    thr = np.partition(y_score, -k)[-k]
    return float(thr)


def f1_at_review_budget(
    y_true: np.ndarray,
    y_score: np.ndarray,
    budget: float,
) -> tuple[float, float, float, float]:
    thr = threshold_from_review_budget(y_score, budget)
    y_pred = (y_score >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )
    return float(thr), float(p), float(r), float(f1)

In [20]:
LAYERS_MAP = {
    "l2_l3": ("layer2", "layer3"),
    "l1_l2_l3": ("layer1", "layer2", "layer3"),
    "l3": ("layer3",),
    "l2": ("layer2",),
}

IMAGE_SIZE_MAP = {
    "224": (224, 224),
    "256": (256, 256),
    "320": (320, 320),
}

In [21]:
def objective(trial: optuna.Trial) -> float:
    t0 = time.time()

    extractor = None
    memory_bank = None

    try:
        # -------------------------
        # Soft params
        # -------------------------
        soft_train_fraction = trial.suggest_float(
            "soft_train_fraction",
            SOFT_TRAIN_FRACTION_RANGE[0],
            SOFT_TRAIN_FRACTION_RANGE[1],
            step=0.01,
        )
        soft_corruption_level = trial.suggest_categorical(
            "soft_corruption_level",
            SOFT_CORRUPTION_LEVELS,
        )
        soft_review_budget = trial.suggest_categorical(
            "soft_review_budget",
            SOFT_REVIEW_BUDGETS,
        )

        # -------------------------
        # PatchCore params
        # -------------------------
        backbone = trial.suggest_categorical(
            "backbone",
            ["wide_resnet50_2", "resnet18", "resnet34", "resnet50"],
        )

        layers_key = trial.suggest_categorical("layers_key", list(LAYERS_MAP.keys()))
        layers = LAYERS_MAP[layers_key]

        pre_trained = trial.suggest_categorical("pre_trained", [True, False])

        image_size_key = trial.suggest_categorical("image_size_key", list(IMAGE_SIZE_MAP.keys()))
        image_size = IMAGE_SIZE_MAP[image_size_key]

        center_crop_key = trial.suggest_categorical("center_crop_key", ["none", "0.875"])
        if center_crop_key == "none":
            center_crop_size = None
        elif center_crop_key == "0.875":
            s = image_size[0]
            center_crop_size = (int(0.875 * s), int(0.875 * s))
        else:
            raise ValueError(center_crop_key)

        coreset_sampling_ratio = trial.suggest_float(
            "coreset_sampling_ratio",
            0.001,
            0.101,
            step=0.002,
        )
        num_neighbors = trial.suggest_int("num_neighbors", 1, 30)
        reduction = trial.suggest_categorical("reduction", ["max", "mean"])
        max_patches_per_image = trial.suggest_categorical(
            "max_patches_per_image",
            [128, 256, 512, 1024],
        )
        batch_size = trial.suggest_categorical("batch_size", [8, 16])

        class_f1s = []
        class_aurocs = []
        per_class_metrics = {}

        for cls in SELECTED_CLASSES:
            df_train_good = CLASS_DATA[cls]["df_train_good"]
            df_val = CLASS_DATA[cls]["df_val"]
            df_final = CLASS_DATA[cls]["df_final"]

            if soft_train_fraction >= 1.0:
                df_sub = df_train_good
            else:
                df_sub = df_train_good.sample(
                    frac=soft_train_fraction,
                    random_state=SEED + trial.number,
                )

            train_loader, val_loader, _ = make_loaders(
                df_train_good_sub=df_sub,
                df_val=df_val,
                df_final=df_final,
                image_size=image_size,
                center_crop_size=center_crop_size,
                batch_size=batch_size,
                corruption_level=soft_corruption_level,
            )

            extractor = FeatureExtractor(
                backbone=backbone,
                layers=layers,
                pre_trained=pre_trained,
            )

            memory_bank = fit_memory_bank(
                extractor=extractor,
                train_loader=train_loader,
                coreset_sampling_ratio=coreset_sampling_ratio,
                max_patches_per_image=max_patches_per_image,
            )
            if len(memory_bank) < num_neighbors:
                raise optuna.TrialPruned(
                    f"memory_bank too small: {len(memory_bank)} < num_neighbors={num_neighbors}"
                )

            y_true_val, y_score_val = score_images(
                extractor=extractor,
                loader=val_loader,
                memory_bank=memory_bank,
                num_neighbors=num_neighbors,
                reduction=reduction,
                max_patches_per_image=max_patches_per_image,
            )

            thr, p, r, f1 = f1_at_review_budget(
                y_true=y_true_val,
                y_score=y_score_val,
                budget=soft_review_budget,
            )
            auroc = roc_auc_score(y_true_val, y_score_val)

            class_f1s.append(float(f1))
            class_aurocs.append(float(auroc))

            per_class_metrics[cls] = {
                "threshold_val_budget": float(thr),
                "precision_val_at_budget": float(p),
                "recall_val_at_budget": float(r),
                "f1_val_at_budget": float(f1),
                "auroc_val": float(auroc),
                "train_good_used": int(len(df_sub)),
                "memory_bank_size": int(memory_bank.shape[0]),
            }

            del extractor, memory_bank, train_loader, val_loader
            extractor = None
            memory_bank = None

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()

        mean_f1 = float(np.mean(class_f1s))
        mean_auroc = float(np.mean(class_aurocs))
        runtime_sec = time.time() - t0

        trial.set_user_attr("selected_classes", SELECTED_CLASSES)
        trial.set_user_attr("mean_f1_val_at_budget", mean_f1)
        trial.set_user_attr("mean_auroc_val", mean_auroc)
        trial.set_user_attr("runtime_sec", float(runtime_sec))
        trial.set_user_attr("per_class_metrics", per_class_metrics)
        trial.set_user_attr("oom", False)

        return mean_f1

    except torch.OutOfMemoryError:
        trial.set_user_attr("oom", True)

        if extractor is not None:
            del extractor
        if memory_bank is not None:
            del memory_bank

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        raise

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [22]:
DATASET_NAME = "mvtec_4class_patchcore"
TARGET_TRIALS = 10000

OUT_DIR = ROOT / "fast" / "x_peer_pdt_hyperoptxai" / "hpo_mvtec_4class" / "outputs_patchcore"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = OUT_DIR / f"{DATASET_NAME}_hpo_logs.db"
STORAGE = f"sqlite:///{DB_PATH.as_posix()}"

print("OUT_DIR:", OUT_DIR)
print("DB_PATH:", DB_PATH)
print("STORAGE:", STORAGE)

OUT_DIR: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/outputs_patchcore
DB_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/outputs_patchcore/mvtec_4class_patchcore_hpo_logs.db
STORAGE: sqlite:////home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/outputs_patchcore/mvtec_4class_patchcore_hpo_logs.db


In [23]:
sampler = optuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=4000,
    multivariate=False,
    group=False,
)

study = optuna.create_study(
    direction="maximize",
    study_name=f"{DATASET_NAME}_{'_'.join(SELECTED_CLASSES)}_v1.0.0",
    storage=STORAGE,
    load_if_exists=True,
    sampler=sampler,
)

print("Study ready:", study.study_name)
print("Trials already stored:", len(study.trials))

[I 2026-03-11 11:10:36,261] Using an existing study with name 'mvtec_4class_patchcore_bottle_cable_capsule_hazelnut_v1.0.0' instead of creating a new one.


Study ready: mvtec_4class_patchcore_bottle_cable_capsule_hazelnut_v1.0.0
Trials already stored: 7262


In [ ]:
already_done = len(study.trials)
remaining = max(TARGET_TRIALS - already_done, 0)

if remaining:
    print(f"Running {remaining} additional trials (have {already_done}, need {TARGET_TRIALS})")
    study.optimize(
        objective,
        n_trials=remaining,
        gc_after_trial=True,
        catch=(torch.OutOfMemoryError,),
    )
else:
    print(f"Target of {TARGET_TRIALS} trials already reached – nothing to do")

print("Best mean F1 (val @ budget):", study.best_value)
print("Best params:", study.best_params)
print("DB:", DB_PATH)

Running 2738 additional trials (have 7262, need 10000)


[I 2026-03-11 11:11:12,474] Trial 7262 finished with value: 0.2265154927040173 and parameters: {'soft_train_fraction': 0.8500000000000001, 'soft_corruption_level': 'none', 'soft_review_budget': 0.08, 'backbone': 'wide_resnet50_2', 'layers_key': 'l2_l3', 'pre_trained': True, 'image_size_key': '256', 'center_crop_key': '0.875', 'coreset_sampling_ratio': 0.023, 'num_neighbors': 4, 'reduction': 'mean', 'max_patches_per_image': 1024, 'batch_size': 16}. Best is trial 4545 with value: 0.8202491952491952.
[I 2026-03-11 11:11:42,607] Trial 7263 finished with value: 0.8142968142968142 and parameters: {'soft_train_fraction': 0.8900000000000001, 'soft_corruption_level': 'none', 'soft_review_budget': 0.495, 'backbone': 'wide_resnet50_2', 'layers_key': 'l2_l3', 'pre_trained': True, 'image_size_key': '256', 'center_crop_key': '0.875', 'coreset_sampling_ratio': 0.031, 'num_neighbors': 1, 'reduction': 'mean', 'max_patches_per_image': 1024, 'batch_size': 16}. Best is trial 4545 with value: 0.82024919524

In [ ]:
best_trial = study.best_trial

print("Best trial number:", best_trial.number)
print("Best objective value:", best_trial.value)
print("\nBest params:")
for k, v in best_trial.params.items():
    print(f"  {k}: {v}")

print("\nBest trial attrs:")
for k, v in best_trial.user_attrs.items():
    print(f"  {k}: {v}")

In [ ]:
def evaluate_best_on_final(best_params: dict) -> pd.DataFrame:
    layers = LAYERS_MAP[best_params["layers_key"]]
    image_size = IMAGE_SIZE_MAP[best_params["image_size_key"]]

    if best_params["center_crop_key"] == "none":
        center_crop_size = None
    else:
        s = image_size[0]
        center_crop_size = (int(0.875 * s), int(0.875 * s))

    rows = []

    for cls in SELECTED_CLASSES:
        df_train_good = CLASS_DATA[cls]["df_train_good"]
        df_val = CLASS_DATA[cls]["df_val"]
        df_final = CLASS_DATA[cls]["df_final"]

        soft_train_fraction = best_params["soft_train_fraction"]
        if soft_train_fraction >= 1.0:
            df_sub = df_train_good
        else:
            df_sub = df_train_good.sample(frac=soft_train_fraction, random_state=SEED)

        train_loader, _, final_loader = make_loaders(
            df_train_good_sub=df_sub,
            df_val=df_val,
            df_final=df_final,
            image_size=image_size,
            center_crop_size=center_crop_size,
            batch_size=best_params["batch_size"],
            corruption_level=best_params["soft_corruption_level"],
        )

        extractor = FeatureExtractor(
            backbone=best_params["backbone"],
            layers=layers,
            pre_trained=best_params["pre_trained"],
        )

        memory_bank = fit_memory_bank(
            extractor=extractor,
            train_loader=train_loader,
            coreset_sampling_ratio=best_params["coreset_sampling_ratio"],
            max_patches_per_image=best_params["max_patches_per_image"],
        )

        y_true_final, y_score_final = score_images(
            extractor=extractor,
            loader=final_loader,
            memory_bank=memory_bank,
            num_neighbors=best_params["num_neighbors"],
            reduction=best_params["reduction"],
            max_patches_per_image=best_params["max_patches_per_image"],
        )

        thr, p, r, f1 = f1_at_review_budget(
            y_true=y_true_final,
            y_score=y_score_final,
            budget=best_params["soft_review_budget"],
        )
        auroc = roc_auc_score(y_true_final, y_score_final)

        rows.append({
            "mvtec_class": cls,
            "precision_final_at_budget": float(p),
            "recall_final_at_budget": float(r),
            "f1_final_at_budget": float(f1),
            "auroc_final": float(auroc),
            "threshold_final_budget": float(thr),
            "train_good_used": int(len(df_sub)),
        })

        del extractor, memory_bank
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)

In [ ]:
df_final_results = evaluate_best_on_final(study.best_params)
df_final_results

In [ ]:
print("Mean final F1:", df_final_results["f1_final_at_budget"].mean())
print("Mean final AUROC:", df_final_results["auroc_final"].mean())

In [ ]:
FINAL_RESULTS_PATH = OUT_DIR / "best_params_final_results.csv"
BEST_PARAMS_PATH = OUT_DIR / "best_params.json"

df_final_results.to_csv(FINAL_RESULTS_PATH, index=False)

with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(study.best_params, f, indent=2)

print("Saved final results to:", FINAL_RESULTS_PATH)
print("Saved best params to:", BEST_PARAMS_PATH)